# Notebook 08: Fine-Tuning and Model Serving

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/08-fine-tuning-serving.ipynb)

## Learning Objectives

By the end of this notebook you will be able to:

1. Prepare training data in JSONL format for OpenAI fine-tuning
2. Validate datasets for completeness, formatting, and quality
3. Create and monitor a fine-tuning job through the OpenAI API
4. Compare base vs fine-tuned model performance
5. Build an A/B testing framework for model evaluation
6. Estimate and analyze fine-tuning costs

---
## 1. Setup and Installation

In [ ]:
!pip install openai --quiet

---
## 2. Configuration

In [ ]:
import os
import io
import json
import random
import hashlib
import math
import time
from collections import Counter
from dataclasses import dataclass
from typing import Optional
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Mock responses will be used.")
    print("Set it with: os.environ['OPENAI_API_KEY'] = 'your-key-here'")

client = OpenAI(api_key=api_key or "sk-mock-key")
MODEL = "gpt-4o-mini"

---
## 3. Training Data Preparation (JSONL Format)

Fine-tuning requires training data in JSONL format -- one JSON object per line.
Each example contains a `messages` array with system, user, and assistant roles.

In [ ]:
CATEGORIES = ["billing", "technical", "account", "shipping", "returns"]

TEMPLATES = {
    "billing": [
        "I was charged twice for my last order.",
        "Can you explain the charges on my invoice?",
        "Why is my bill higher than expected this month?",
        "I need a refund for an unauthorized charge.",
        "How do I update my payment method?",
        "My coupon code didn't apply the discount.",
    ],
    "technical": [
        "The app crashes when I try to upload a photo.",
        "I can't log in even though my password is correct.",
        "The website is very slow today.",
        "How do I integrate your API with my application?",
        "Getting a 500 error when submitting the form.",
        "The search feature returns no results.",
    ],
    "account": [
        "How do I change my email address?",
        "I want to delete my account permanently.",
        "Can I merge two accounts together?",
        "How do I enable two-factor authentication?",
        "I forgot my username, how do I recover it?",
        "Can I change my subscription plan?",
    ],
    "shipping": [
        "Where is my package? It was supposed to arrive yesterday.",
        "Do you ship internationally?",
        "How long does standard shipping take?",
        "My tracking number doesn't work.",
        "Can I change the delivery address after ordering?",
        "The package arrived damaged.",
    ],
    "returns": [
        "I want to return an item I bought last week.",
        "What is your return policy?",
        "How do I get a prepaid return label?",
        "Can I exchange this for a different size?",
        "How long does it take to process a return?",
        "I received the wrong item and need to return it.",
    ],
}

SYSTEM_PROMPT = ("You are a customer support classifier. Given a customer message, "
                 "respond with exactly one category: billing, technical, account, "
                 "shipping, or returns.")


def generate_training_data() -> list[dict]:
    """Generate JSONL-format training examples."""
    examples = []
    for category, messages in TEMPLATES.items():
        for msg in messages:
            examples.append({
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": msg},
                    {"role": "assistant", "content": category},
                ]
            })
    random.seed(42)
    random.shuffle(examples)
    return examples


raw_data = generate_training_data()
print(f"Generated {len(raw_data)} training examples")
print(f"\nExample JSONL entry:")
print(json.dumps(raw_data[0], indent=2))

# Distribution
labels = [ex["messages"][2]["content"] for ex in raw_data]
dist = Counter(labels)
print(f"\nCategory distribution:")
for cat, count in sorted(dist.items()):
    print(f"  {cat:<12} {count:>3} {'#' * count}")

---
## 4. Data Validation

Before fine-tuning, validate the dataset for proper formatting, remove
duplicates, check lengths, and filter low-quality examples.

In [ ]:
@dataclass
class QualityReport:
    total_input: int
    total_output: int
    removed_short: int
    removed_long: int
    removed_duplicates: int
    removed_format: int
    errors: list


class DataValidator:
    """Validate and clean fine-tuning data."""

    REQUIRED_ROLES = {"system", "user", "assistant"}

    def __init__(self, min_user_len=5, max_user_len=2000):
        self.min_user_len = min_user_len
        self.max_user_len = max_user_len

    def validate_and_clean(self, data: list[dict]) -> tuple[list[dict], QualityReport]:
        errors = []
        rm_short = rm_long = rm_dup = rm_fmt = 0
        seen_hashes = set()
        clean = []

        for i, ex in enumerate(data):
            # Format check
            msgs = ex.get("messages", [])
            roles = {m.get("role") for m in msgs}
            if not self.REQUIRED_ROLES.issubset(roles):
                rm_fmt += 1
                errors.append(f"Row {i}: missing roles {self.REQUIRED_ROLES - roles}")
                continue

            user_text = next((m["content"] for m in msgs if m["role"] == "user"), "")

            # Length
            if len(user_text) < self.min_user_len:
                rm_short += 1; continue
            if len(user_text) > self.max_user_len:
                rm_long += 1; continue

            # Dedup
            h = hashlib.md5(user_text.lower().strip().encode()).hexdigest()
            if h in seen_hashes:
                rm_dup += 1; continue
            seen_hashes.add(h)

            clean.append(ex)

        report = QualityReport(
            total_input=len(data), total_output=len(clean),
            removed_short=rm_short, removed_long=rm_long,
            removed_duplicates=rm_dup, removed_format=rm_fmt,
            errors=errors[:5])
        return clean, report


# Add some bad examples
bad = [
    {"messages": [{"role": "user", "content": "Hi"}]},             # Missing roles
    {"messages": [{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": "Hi"},
                  {"role": "assistant", "content": "billing"}]},   # Too short
    raw_data[0],                                                     # Duplicate
]

validator = DataValidator()
clean_data, report = validator.validate_and_clean(raw_data + bad)

print("Data Quality Report")
print("=" * 40)
print(f"  Input:     {report.total_input}")
print(f"  Output:    {report.total_output}")
print(f"  Rm short:  {report.removed_short}")
print(f"  Rm long:   {report.removed_long}")
print(f"  Rm dupes:  {report.removed_duplicates}")
print(f"  Rm format: {report.removed_format}")
print(f"  Retention: {report.total_output/report.total_input:.1%}")
if report.errors:
    print(f"  Errors: {report.errors}")

---
## 5. Fine-Tuning Job Creation (OpenAI API)

This section demonstrates the full fine-tuning workflow: upload a training file,
create a job, and poll for completion. Mock fallbacks are used without an API key.

In [ ]:
def create_jsonl_bytes(data: list[dict]) -> bytes:
    """Convert training data to JSONL bytes for upload."""
    lines = [json.dumps(ex) for ex in data]
    return "\n".join(lines).encode("utf-8")


def upload_training_file(data: list[dict]) -> str:
    """Upload JSONL training data to OpenAI."""
    try:
        content = create_jsonl_bytes(data)
        result = client.files.create(
            file=("training.jsonl", io.BytesIO(content)),
            purpose="fine-tune"
        )
        file_id = result.id
    except Exception as e:
        print(f"API call failed: {e}")
        file_id = "file-mock-abc123"
        print("Using mock response for demonstration.")
    print(f"Uploaded file: {file_id}")
    return file_id


def create_fine_tuning_job(file_id: str,
                           base_model: str = "gpt-4o-mini-2024-07-18",
                           n_epochs: int = 3) -> dict:
    """Create a fine-tuning job."""
    try:
        job = client.fine_tuning.jobs.create(
            training_file=file_id,
            model=base_model,
            hyperparameters={"n_epochs": n_epochs}
        )
        info = {"id": job.id, "model": job.model, "status": job.status}
    except Exception as e:
        print(f"API call failed: {e}")
        info = {"id": "ftjob-mock-xyz789", "model": base_model,
                "status": "validating_files"}
        print("Using mock response for demonstration.")
    print(f"Fine-tuning job: {json.dumps(info, indent=2)}")
    return info


# Run the workflow
file_id = upload_training_file(clean_data)
job_info = create_fine_tuning_job(file_id, n_epochs=3)

---
## 6. Monitoring Training Metrics

Track training progress by polling job status and retrieving training metrics
like loss over epochs.

In [ ]:
def monitor_job(job_id: str) -> dict:
    """Poll fine-tuning job status and retrieve metrics."""
    try:
        job = client.fine_tuning.jobs.retrieve(job_id)
        status = {"id": job.id, "status": job.status,
                  "model": getattr(job, "fine_tuned_model", None)}
        # Get events
        events = client.fine_tuning.jobs.list_events(job_id, limit=10)
        status["events"] = [{"message": e.message, "created": e.created_at}
                            for e in events.data]
    except Exception as e:
        print(f"API call failed: {e}")
        print("Using mock response for demonstration.")
        status = {
            "id": job_id, "status": "succeeded",
            "model": "ft:gpt-4o-mini-2024-07-18:org::mock123",
            "events": [
                {"message": "Created fine-tuning job",        "created": 1700000000},
                {"message": "Validating training file",       "created": 1700000010},
                {"message": "Training started",               "created": 1700000060},
                {"message": "Step 100: training loss=0.45",   "created": 1700000300},
                {"message": "Step 200: training loss=0.22",   "created": 1700000600},
                {"message": "Step 300: training loss=0.11",   "created": 1700000900},
                {"message": "Training completed",             "created": 1700001000},
                {"message": "Model ft:gpt-4o-mini-2024-07-18:org::mock123 ready",
                 "created": 1700001010},
            ]
        }

    print(f"Job: {status['id']}")
    print(f"Status: {status['status']}")
    print(f"Fine-tuned model: {status.get('model', 'N/A')}")
    print(f"\nRecent events:")
    for evt in status["events"][-6:]:
        print(f"  {evt['message']}")

    # Parse loss from events
    losses = []
    for evt in status["events"]:
        msg = evt["message"]
        if "training loss" in msg:
            import re
            m = re.search(r"loss[=:]\s*([\d.]+)", msg)
            if m:
                losses.append(float(m.group(1)))

    if losses:
        print(f"\nTraining loss curve: {losses}")
        print(f"  Start: {losses[0]:.4f}  End: {losses[-1]:.4f}  "
              f"Reduction: {(1-losses[-1]/losses[0])*100:.1f}%")
    return status


status = monitor_job(job_info["id"])

---
## 7. Model Comparison (Base vs Fine-Tuned)

Compare the base model and the fine-tuned model on the same evaluation set
to measure accuracy improvement.

In [ ]:
def classify_with_model(model_name: str, user_text: str) -> str:
    """Classify a single example using the specified model."""
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_text},
            ],
            temperature=0, max_tokens=10
        )
        result = response.choices[0].message.content.strip().lower()
    except Exception as e:
        result = None
    return result


def evaluate_model(model_name: str, eval_set: list[dict],
                   mock_accuracy: float = 0.7) -> dict:
    """Evaluate a model on the eval set. Uses mock when API unavailable."""
    predictions, references = [], []
    random.seed(hash(model_name) % 10000)

    for ex in eval_set:
        user_text = next(m["content"] for m in ex["messages"] if m["role"] == "user")
        true_label = ex["messages"][2]["content"]
        references.append(true_label)

        pred = classify_with_model(model_name, user_text)
        if pred is None or pred not in CATEGORIES:
            # Mock fallback
            if random.random() < mock_accuracy:
                pred = true_label
            else:
                pred = random.choice([c for c in CATEGORIES if c != true_label])
        predictions.append(pred)

    correct = sum(p == r for p, r in zip(predictions, references))
    total = len(references)

    # Per-class F1
    per_class = {}
    for cls in CATEGORIES:
        tp = sum(1 for p, r in zip(predictions, references) if p == cls and r == cls)
        fp = sum(1 for p, r in zip(predictions, references) if p == cls and r != cls)
        fn = sum(1 for p, r in zip(predictions, references) if p != cls and r == cls)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        per_class[cls] = {"precision": round(prec, 3), "recall": round(rec, 3),
                          "f1": round(f1, 3)}

    return {"model": model_name, "accuracy": round(correct / total, 4),
            "correct": correct, "total": total, "per_class": per_class}


# Create train/eval split
random.seed(42)
shuffled = clean_data.copy()
random.shuffle(shuffled)
split = int(len(shuffled) * 0.8)
train_set, eval_set = shuffled[:split], shuffled[split:]
print(f"Train: {len(train_set)}  Eval: {len(eval_set)}")

# Compare models
base_result = evaluate_model("gpt-4o-mini", eval_set, mock_accuracy=0.70)
ft_result   = evaluate_model("ft:gpt-4o-mini:mock", eval_set, mock_accuracy=0.92)

print(f"\n{'Model':<30} {'Accuracy':>10}")
print("-" * 42)
print(f"{base_result['model']:<30} {base_result['accuracy']:>9.1%}")
print(f"{ft_result['model']:<30} {ft_result['accuracy']:>9.1%}")

delta = ft_result['accuracy'] - base_result['accuracy']
print(f"\nImprovement: {delta:+.1%}")

print(f"\nPer-class F1:")
print(f"{'Class':<12} {'Base':>8} {'FT':>8} {'Delta':>8}")
print("-" * 38)
for cls in CATEGORIES:
    b = base_result["per_class"].get(cls, {}).get("f1", 0)
    f = ft_result["per_class"].get(cls, {}).get("f1", 0)
    print(f"{cls:<12} {b:>8.3f} {f:>8.3f} {f-b:>+8.3f}")

---
## 8. A/B Testing Framework

Route a percentage of traffic to the fine-tuned model and compare live
performance metrics.

In [ ]:
class ABTestRouter:
    """Route requests between two models for A/B testing."""

    def __init__(self, model_a: str, model_b: str, b_fraction: float = 0.2):
        self.model_a = model_a
        self.model_b = model_b
        self.b_fraction = b_fraction
        self.results_a: list[dict] = []
        self.results_b: list[dict] = []

    def route(self, user_text: str, true_label: str = None) -> dict:
        use_b = random.random() < self.b_fraction
        model = self.model_b if use_b else self.model_a

        t0 = time.time()
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_text},
                ],
                temperature=0, max_tokens=10
            )
            pred = response.choices[0].message.content.strip().lower()
        except Exception:
            # Mock
            acc = 0.92 if use_b else 0.70
            if true_label and random.random() < acc:
                pred = true_label
            else:
                pred = random.choice(CATEGORIES)
        latency = time.time() - t0

        entry = {"model": model, "prediction": pred,
                 "true_label": true_label, "latency_ms": round(latency * 1000, 1),
                 "correct": pred == true_label if true_label else None}
        (self.results_b if use_b else self.results_a).append(entry)
        return entry

    def report(self) -> dict:
        def _stats(results):
            n = len(results)
            if n == 0:
                return {"n": 0}
            correct = sum(1 for r in results if r["correct"])
            avg_lat = sum(r["latency_ms"] for r in results) / n
            return {"n": n, "accuracy": round(correct / n, 3),
                    "avg_latency_ms": round(avg_lat, 1)}
        return {"model_a": _stats(self.results_a),
                "model_b": _stats(self.results_b)}


# Simulate A/B test
random.seed(42)
router = ABTestRouter("gpt-4o-mini", "ft:gpt-4o-mini:mock", b_fraction=0.3)

for ex in eval_set:
    user_text = next(m["content"] for m in ex["messages"] if m["role"] == "user")
    true_label = ex["messages"][2]["content"]
    router.route(user_text, true_label)

report = router.report()
print("A/B Test Report")
print("=" * 50)
print(f"  Model A (base):      {json.dumps(report['model_a'])}")
print(f"  Model B (fine-tuned): {json.dumps(report['model_b'])}")

if report['model_b']['n'] > 0 and report['model_a']['n'] > 0:
    d = report['model_b']['accuracy'] - report['model_a']['accuracy']
    print(f"\n  Accuracy delta: {d:+.3f}")
    if d > 0.05:
        print("  Recommendation: DEPLOY fine-tuned model")
    else:
        print("  Recommendation: Keep base model (improvement marginal)")

---
## 9. Cost Analysis

Estimate fine-tuning and inference costs based on dataset size,
token counts, and pricing.

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough token estimate (1 token ~ 4 chars for English)."""
    return max(1, len(text) // 4)


def cost_analysis(data: list[dict], n_epochs: int = 3) -> dict:
    """Estimate fine-tuning cost based on OpenAI pricing."""
    total_tokens = 0
    for ex in data:
        for msg in ex["messages"]:
            total_tokens += estimate_tokens(msg["content"])

    training_tokens = total_tokens * n_epochs

    # Approximate OpenAI fine-tuning pricing (as of 2024)
    cost_per_1k_train = 0.008  # gpt-4o-mini fine-tuning
    cost_per_1k_infer = 0.0003 # gpt-4o-mini inference (input)

    train_cost = (training_tokens / 1000) * cost_per_1k_train

    # Break-even: how many inference calls before FT cost is recovered
    # (assuming FT model is more accurate and reduces re-queries)
    avg_infer_tokens = total_tokens / len(data)
    infer_cost_per_call = (avg_infer_tokens / 1000) * cost_per_1k_infer

    return {
        "n_examples": len(data),
        "total_tokens": f"{total_tokens:,}",
        "training_tokens": f"{training_tokens:,}",
        "n_epochs": n_epochs,
        "estimated_train_cost": f"${train_cost:.2f}",
        "cost_per_inference": f"${infer_cost_per_call:.6f}",
    }


analysis = cost_analysis(clean_data, n_epochs=3)
print("Cost Analysis")
print("=" * 40)
for k, v in analysis.items():
    print(f"  {k:<25} {v}")

# Compare different dataset sizes
print(f"\n{'Examples':>10} {'Epochs':>8} {'Train Tokens':>15} {'Cost':>10}")
print("-" * 48)
for n in [30, 100, 500, 1000, 5000]:
    # Simulate n examples
    mock_data = clean_data * (n // len(clean_data) + 1)
    mock_data = mock_data[:n]
    a = cost_analysis(mock_data, n_epochs=3)
    print(f"{n:>10} {3:>8} {a['training_tokens']:>15} {a['estimated_train_cost']:>10}")

---
## Summary

This notebook covered the complete fine-tuning and model serving workflow:

1. **Training Data Preparation** -- Generate and format JSONL data with system/user/assistant messages.
2. **Data Validation** -- Check format, remove short/long/duplicate examples, and report quality.
3. **Fine-Tuning Job Creation** -- Upload data and create jobs via the OpenAI API.
4. **Monitoring Training** -- Poll status, retrieve events, and parse loss curves.
5. **Model Comparison** -- Evaluate base vs fine-tuned accuracy and per-class F1.
6. **A/B Testing** -- Route traffic to compare models with live metrics.
7. **Cost Analysis** -- Estimate training costs and per-inference pricing.

### Key Takeaways

- Always validate data quality before fine-tuning.
- Start with a small dataset and iterate -- more data is not always better.
- Use A/B testing to make deployment decisions based on real metrics.
- Fine-tuning is most valuable for domain-specific tasks where the base model underperforms.